# Compendium Data Pipeline (Simplified)

Loops through each chapter folder (Health, Population, Education, Labor,
Poverty, Housing) under `datacollector_received_quest`, reshapes and merges
every Excel file's data/source tables, corrects the Arabic labels against
`translation dict.xlsx`, translates to English, and saves per-chapter
`<Chapter>_AR.xlsx` / `<Chapter>_EN.xlsx` outputs to the COMPENDIUM-ARAB
SOCIETY folder.

This version favors simplicity over defensiveness: plain functions (no
classes), one try/except per sheet, and flat, plainly-named dictionaries.

In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium_pipeline")


## Config / paths

In [ ]:
"""
CELL: Configuration - paths, the list of chapters, and the fuzzy-match cutoff.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
RECEIVED_QUEST_PATH = DATA_COLLECTOR_PATH / "datacollector_received_quest"
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
OUTPUT_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

CHAPTERS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]

# A fuzzy match must score at least this well (0 to 1) to be used.
# Below this, we leave the value alone and just log a warning, instead of
# guessing and possibly writing something wrong.
FUZZY_MATCH_CUTOFF = 0.6

# Columns used to attach the right source/citation row to each data row:
# year, indicator, country.
MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]


## Load the translation dictionary

Reads `translation dict.xlsx` once and turns it into two simple dictionaries:
one for renaming columns, one for renaming values (per column).

In [ ]:
"""
CELL: Load translation dict.xlsx into two plain lookup dictionaries.
"""


def load_dictionary():
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_translations = {}  # Arabic column name -> English column name
    value_translations = {}   # Arabic column name -> {Arabic value: English value}

    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_translations[arabic_column] = rows["col_en"].iloc[0]
        value_translations[arabic_column] = {
            ar: en for ar, en in zip(rows["val_ar"], rows["val_en"]) if pd.notna(ar)
        }

    # The dictionary also has a "Chapter" column (col_en == "Chapter") whose
    # rows are exactly the chapter names, e.g. "Labor" <-> "عمالة".
    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return column_translations, value_translations, chapter_to_arabic


COLUMN_TRANSLATIONS, VALUE_TRANSLATIONS, CHAPTER_TO_ARABIC = load_dictionary()
logger.info(f"Dictionary loaded: {len(COLUMN_TRANSLATIONS)} Arabic columns available")


## `extract_tables()`

Each sheet has two tables marked by an `index` column: `index=1` is the data
table, `index=2` is the source table. A row where column 0 says `"index"`
holds the column names for the table that follows it.

In [ ]:
"""
CELL: extract_tables() - split one raw sheet into its data table and source table.
"""


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    data_columns = raw_sheet.iloc[data_header_row].dropna()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


## `reshape_and_merge()`

Unpivots the data table's year columns (any column whose name is all
digits) into two columns (year, value), then merges in the matching row
from the source table so every data point carries its source.

In [ ]:
"""
CELL: reshape_and_merge() - wide-to-long reshape, then attach the source table.
"""


def reshape_and_merge(data_table, source_table):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="السنة",   # "Year"
        value_name="العدد",  # "Value"
    )

    merge_columns = [c for c in MERGE_COLUMNS if c in long_table.columns and c in source_table.columns]
    return pd.merge(long_table, source_table, on=merge_columns, how="left")


## `correct_with_dictionary()`

For every column: if its name is already a known Arabic column, leave it.
Otherwise, find the known column name it's most similar to, and rename it
there - but only if that similarity score clears `FUZZY_MATCH_CUTOFF`.

Then do the same thing for every value within each known column: if the
value is already known, leave it; otherwise replace it with the closest
known value, if close enough. Every actual change is printed.

In [ ]:
"""
CELL: correct_with_dictionary() - fix column names and cell values.
"""


def best_match(text, choices):
    """Compares text against every choice and returns (best_choice, score) -
    the one difflib considers most similar, and how similar (0 to 1)."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def correct_with_dictionary(table, chapter, file_name, sheet_name):
    table = table.copy()
    replacements = 0

    # 1. Fix column names.
    for column in list(table.columns):
        if column in COLUMN_TRANSLATIONS:
            continue  # already a known column name, nothing to fix

        match, score = best_match(column, COLUMN_TRANSLATIONS.keys())
        if match is not None and score >= FUZZY_MATCH_CUTOFF:
            print(f"[{chapter}] {file_name} | {sheet_name} | column: {column} -> {match} (score={score:.2f})")
            table = table.rename(columns={column: match})
            replacements += 1
        elif match is not None:
            logger.warning(
                f"[{chapter}] {file_name} | {sheet_name} | column '{column}' has no good match "
                f"(closest is '{match}', score={score:.2f}) - left unchanged"
            )

    # 2. Fix cell values, one known column at a time.
    for column in table.columns:
        known_values = VALUE_TRANSLATIONS.get(column)
        if not known_values:
            continue  # not a dictionary column, or it has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue  # already a known value, nothing to fix

            match, score = best_match(value, known_values.keys())
            if match is not None and score >= FUZZY_MATCH_CUTOFF:
                print(f"[{chapter}] {file_name} | {sheet_name} | {column}: {value} -> {match} (score={score:.2f})")
                table[column] = table[column].replace(value, match)
                replacements += 1
            elif match is not None:
                logger.warning(
                    f"[{chapter}] {file_name} | {sheet_name} | {column} value '{value}' has no good match "
                    f"(closest is '{match}', score={score:.2f}) - left unchanged"
                )

    return table, replacements


## `translate()`

Swaps every Arabic value for its English equivalent, then renames the
column to its English name - using the same two dictionaries built above.

In [ ]:
"""
CELL: translate() - Arabic to English, using the same dictionaries.
"""


def translate(table):
    table = table.copy()
    for column in list(table.columns):
        if column in VALUE_TRANSLATIONS:
            table[column] = table[column].replace(VALUE_TRANSLATIONS[column])
        if column in COLUMN_TRANSLATIONS:
            table = table.rename(columns={column: COLUMN_TRANSLATIONS[column]})
    return table


## Error tracking

`log_failure()` is the one place that logs a failure and remembers it, so
the final cell can print one consolidated summary grouped by chapter.

In [ ]:
"""
CELL: Error tracking - log a failure and remember it for the run summary.
"""
FAILURES = defaultdict(list)  # chapter -> list of {file, sheet, step, error}

# Plain-language guess at what's wrong, keyed by which step failed.
LIKELY_CAUSES = {
    "read": "file may be corrupted, password-protected, or not a valid .xlsx",
    "extract": "sheet may be missing an 'index' column, or index values are not 1/2 as expected",
    "reshape_and_merge": "data table may be missing year columns, or the merge columns don't match the source table",
    "dictionary correction": "column names or values may be malformed and unable to be matched",
    "translation": "a column or value may not have a corresponding English entry in the dictionary",
}


def log_failure(chapter, file_name, sheet_name, step, error):
    likely_cause = LIKELY_CAUSES.get(step, "unexpected error, inspect the sheet manually")
    logger.error(
        f"[ERROR] Chapter={chapter} | File={file_name} | Sheet={sheet_name} | "
        f"Step={step} | {type(error).__name__}: {error} | Likely cause: {likely_cause}"
    )
    FAILURES[chapter].append({
        "file": file_name,
        "sheet": sheet_name,
        "step": step,
        "error": f"{type(error).__name__}: {error}",
    })


## `process_chapter()`

Runs every step, for every sheet, for every file in one chapter. Each sheet
is processed inside one try/except; a plain `step` variable tracks which
stage we're at, so a failure is logged with the right step name without
needing a separate try/except per step.

In [ ]:
"""
CELL: process_chapter() - ties every step together for one chapter, and saves the result.
"""


def process_chapter(chapter):
    folder = RECEIVED_QUEST_PATH / chapter
    if not folder.exists():
        logger.warning(f"Chapter folder not found, skipping: {folder}")
        return

    files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    logger.info(f"Processing {chapter}: {len(files)} file(s) found")

    arabic_tables = []
    english_tables = []
    replacements_made = 0

    for file_path in files:
        file_name = file_path.name
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            log_failure(chapter, file_name, "-", "read", error)
            continue

        logger.info(f"  {chapter}/{file_name}: {len(xls.sheet_names)} sheet(s)")

        for sheet_name in xls.sheet_names:
            step = "read"
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)

                step = "extract"
                data_table, source_table = extract_tables(raw_sheet)
                data_table["الفصل"] = CHAPTER_TO_ARABIC[chapter]  # "Chapter"

                step = "reshape_and_merge"
                merged_table = reshape_and_merge(data_table, source_table)

                step = "dictionary correction"
                corrected_table, n = correct_with_dictionary(merged_table, chapter, file_name, sheet_name)
                replacements_made += n

                step = "translation"
                english_table = translate(corrected_table)

            except Exception as error:
                log_failure(chapter, file_name, sheet_name, step, error)
                continue

            arabic_tables.append(corrected_table)
            english_tables.append(english_table)

    logger.info(
        f"{chapter}: {len(arabic_tables)} sheet(s) processed successfully, "
        f"{replacements_made} dictionary replacement(s) made"
    )

    if not arabic_tables:
        logger.warning(f"{chapter}: no data extracted, no output files written")
        return

    # Stack every sheet's rows on top of each other (concatenate, not merge side-by-side).
    arabic_result = pd.concat(arabic_tables, ignore_index=True)
    english_result = pd.concat(english_tables, ignore_index=True)

    arabic_result.to_excel(OUTPUT_PATH / f"{chapter}_AR.xlsx", index=False, engine="openpyxl")
    english_result.to_excel(OUTPUT_PATH / f"{chapter}_EN.xlsx", index=False, engine="openpyxl")
    logger.info(f"{chapter}: saved {chapter}_AR.xlsx and {chapter}_EN.xlsx")


## Run the pipeline for all chapters

In [ ]:
"""
CELL: Main run - processes every chapter and writes the AR/EN files.
"""
FAILURES.clear()

for chapter in CHAPTERS:
    process_chapter(chapter)


## Run summary - failures grouped by chapter

In [ ]:
"""
CELL: Run summary - every failure from the run above, grouped by chapter.
"""
print("\n" + "=" * 70)
print("RUN SUMMARY - FAILURES BY CHAPTER")
print("=" * 70)

if not FAILURES:
    print("No failures. All files/sheets processed successfully.")
else:
    total = sum(len(v) for v in FAILURES.values())
    print(f"{total} failure(s) across {len(FAILURES)} chapter(s):\n")
    for chapter, failures in FAILURES.items():
        print(f"{chapter} ({len(failures)} failure(s)):")
        for f in failures:
            print(f"  - {f['file']} | Sheet={f['sheet']} | Step={f['step']} | {f['error']}")
        print()
